In [1]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
import os
from eval_utils import get_vecser_for_file
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
def absoluteFilePaths(directory):
    file_names = []
    file_paths = []
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            file_names.append(f)
            file_paths.append(os.path.abspath(os.path.join(dirpath, f)))
    return file_names, file_paths

In [4]:
hook_file_names, hook_file_paths = absoluteFilePaths(os.getenv('VAL_HOOK'))

In [5]:
print(hook_file_paths)

['/media/maindisk/data/mel_harm_CA_all/CA_test/13765_regina-spektor_all-the-rowboats_verse.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/15154_sia_elastic-heart_pre-chorus.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/17465_the-troggs_wild-thing_verse.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/10877_midlake_acts-of-man_intro-and-verse.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/11670_neru_the-disease-called-love_intro.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/9591_lindsey-stirling_elements_chorus.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/16510_the-bamboos_the-wilhelm-scream_outro.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/18078_tomoko-sasaki_song-of-courage_verse.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/10495_massive-attack_teardrop_verse.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/8052_judy-garland_over-the-rainbow_chorus.mid', '/media/maindisk/data/mel_harm_CA_all/CA_test/5818_gamma-ray_lake-of-tears_intro.m

In [6]:
device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [7]:
test_vs = get_vecser_for_file(
    hook_file_paths[0],
    tokenizer,
    graph_model=graph_adapter_model,
    token_model=token_adapter_model,
    adapter_model=adapter_model
)

In [8]:
import numpy as np
x = (np.vstack(test_vs['adapter']))

In [10]:
print(x.shape)

(7, 512)


In [11]:
print(test_vs['chord_symbols'])

[['A:min'], ['F:maj', 'D:min'], ['A:min'], ['F:maj', 'D:min'], ['A:min'], ['F:maj', 'D:min'], ['A:min']]
